# 07b · FLUX.1-dev seed extension — three more adapters per arm (seeds 3, 4, 5)

**Pre-specification: `RESULTS.md` Entry 60 (2026-09-15), registered before any adapter is trained.**
This notebook extends the FLUX.1-dev replication (notebook 07, `E_TRACKB2_FLUX`) from three to six
adapters per arm. Everything except the seeds being trained is notebook 07's: recipe, base-study
fingerprints (`K_A_E2`, `K_B_E2`), training crops, cached caption embedding, seed bank
(`"a photograph, sks style"`, generation seeds 770000–770499), generation settings (1024 px,
28 steps, guidance 3.5) and the measurement statistic. The recipe cells are copied from 07's JSON by a
build script that asserts every edit, not retyped.

**What it does**
1. **G · GPU gate.** Prints the GPU name and memory and **halts below 90 GiB free**.
2. **E0 · gates.** Nothing is trained until all four pass:
   (i) the protocol hash computed here equals the `config_sha` stored in all six of 07's
   `train_meta.json` files (and in `E4_flux_results.json`);
   (ii) a read-only snapshot (SHA-256) of 07's adapters, measurement CSV and results JSON, re-verified
   before the analysis;
   (iii) the seed bank: one of 07's adapters regenerates its stored images 00000/00001;
   (iv) the measurement: this notebook's forensic core re-measures four stored v1 generations and
   must match 07's `f3_measure.csv` rows.
3. **F0 → F1 → F2 → F3.** 07's smoke test, training, generation and measurement, restricted to
   `A/B_raw_s{3,4,5}_flux`. New measurements go to `csv/f3_measure_ext.csv`; 07's CSV is only read.
4. **E60 · analysis, exactly as Entry 60 specifies.** Per-arm exact sign-flip p over six adapters
   (floor 1/64); max-arm limit with t₀.₉₉₅,₅; symmetric θ_sym with Welch SE and its one-sided 99 % limit,
   both as fractions of the real-image contrast 07 used for FLUX (R_real = 3.56703e-02, cross-checked
   against `E4_flux_results.json` and `analysis/FINAL_LEDGER.json`). It prints which Entry 60 reading
   applies and writes `E_TRACKB2_FLUX/flux_seed_ext_summary.json`. It refuses to run until all twelve
   adapters are completely measured.

**How the protocol hash is kept.** 07 hashes every `CFG` field except `STAGES`, and `SEEDS` is inside the
hash. So this notebook keeps `CFG` field for field as 07 has it, **including `SEEDS=(0,1,2)`**. That makes
`CFG_SHA` reproduce 07's value (gate E0-i proves it against the stored adapters, so the old adapters
stay valid). The extension lives outside `CFG`: `NEW_SEEDS=(3,4,5)`, bound by a second hash
`EXT_SHA = sha256(CFG_SHA, NEW_SEEDS, "Entry 60")`. Every new adapter records both hashes and must match
both to count as complete. This is how 02 and 12 kept their base adapters valid (a separate root there;
here the same root, with new tags only).

**Never overwrites seeds 0–2.** Every write goes through a guard that refuses 07's tags. The only shared
files it can create are ones 07 would have created identically, and only when they are missing:
`adapters/_prompt.pt` and `train_png/{A,B}`. A partial `train_png` halts.

**Runtime.** 07 took about 13 h on an A100-80GB for 3 + 3 adapters, including 3,000 generations.
The extension is the same amount of work (six adapters and 3,000 images). **Budget about 13 h;
on the RTX PRO 6000 Blackwell (G4) expect roughly 9–13 h. That range is an estimate, not a
measurement.** Add about 15 min for E0 (which includes the ~35 GB model download) and about 20 min for
F3 + E60. Split it across sessions if your Colab session limit is shorter; every stage resumes.

**Requirements**
- **GPU: Colab "G4" (NVIDIA RTX PRO 6000 Blackwell, 96 GB). Do not run this on any GPU with less than
  90 GiB free.** The recipe is 07's no-checkpointing configuration at 1024 px, and `GRAD_CKPT` is
  inside the protocol hash, so it cannot be switched on to fit a smaller card.
- **bf16** is required. The notebook halts on an fp16 or fp32 fallback.
- **Hugging Face token.** FLUX.1-dev is gated: accept its licence (FLUX.1 [dev] Non-Commercial License)
  on huggingface.co/black-forest-labs/FLUX.1-dev, then add the token as the Colab Secret `HF_TOKEN`
  with *Notebook access* enabled.
- **Google Drive.** Mount the account whose `MyDrive` holds `inv_channel/E_INV_P0_v3` (manifest,
  fingerprints) and `inv_channel/E_TRACKB2_FLUX` (07's outputs). If the archive is shared with you,
  add shortcuts at exactly those paths. About 6 GB of new Drive space is needed (3,000 PNGs plus six
  adapters).


In [ ]:
# %% G · GPU gate — RUN FIRST. Halts unless the GPU has >= 90 GiB free.
import os, sys, subprocess
MIN_FREE_GIB = 90.0          # RTX PRO 6000 Blackwell (Colab "G4"): ~95.6 GiB total, ~94 GiB free fresh
ANALYSIS_ONLY_NO_GPU = False # True ONLY to run stage E60 alone on a CPU runtime (no GPU stages allowed)
def _gate_halt(m): raise SystemExit(f"\n{'='*72}\n[GPU GATE] {m}\n{'='*72}")
try:
    import torch
except ImportError:
    _gate_halt("torch is not installed — this must be a Colab GPU runtime")
if ANALYSIS_ONLY_NO_GPU:
    print("[GPU GATE] ANALYSIS_ONLY_NO_GPU=True — gate skipped; only stage E60 may run")
else:
    if not torch.cuda.is_available():
        _gate_halt("no CUDA device. Runtime -> Change runtime type -> G4 (RTX PRO 6000, 96 GB)")
    _free, _total = torch.cuda.mem_get_info(0)
    _name = torch.cuda.get_device_name(0); _cap = torch.cuda.get_device_capability(0)
    print(f"[GPU GATE] {_name}  compute capability {_cap[0]}.{_cap[1]}")
    print(f"[GPU GATE] memory: {_free/2**30:.1f} GiB free / {_total/2**30:.1f} GiB total")
    print(f"[GPU GATE] torch {torch.__version__}  CUDA {torch.version.cuda}  "
          f"bf16 supported: {torch.cuda.is_bf16_supported()}")
    try:
        print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used,driver_version",
                              "--format=csv"], capture_output=True, text=True).stdout.strip())
    except Exception:
        pass
    if "RTX PRO 6000" not in _name:
        print(f"[GPU GATE] note: expected the RTX PRO 6000 Blackwell (G4); got {_name}. "
              "Memory is what is enforced.")
    if _free/2**30 < MIN_FREE_GIB:
        _gate_halt(f"only {_free/2**30:.1f} GiB free (< {MIN_FREE_GIB:.0f} GiB). This notebook must not "
                   "run on a smaller GPU:\nnotebook 07's recipe (1024 px, no gradient checkpointing) "
                   "is fixed by the protocol hash.")
    if not torch.cuda.is_bf16_supported():
        _gate_halt("bf16 not supported on this GPU — the recipe is bf16")
    # a PyTorch wheel without kernels for this architecture fails here, not after the download
    _x = torch.randn(512, 512, device="cuda", dtype=torch.bfloat16)
    _ok = bool(torch.isfinite((_x @ _x).float()).all()); del _x; torch.cuda.empty_cache()
    if not _ok: _gate_halt("bf16 matmul test produced non-finite values")
    print("[GPU GATE] PASS — bf16 kernels run on this device")


In [ ]:
# %% CFG — notebook 07's configuration, copied from its JSON. Only the seeds TRAINED differ.
import os, sys, json, glob, math, time, gc, shutil, hashlib, subprocess, random, copy, itertools, platform
from dataclasses import dataclass, asdict, replace
from concurrent.futures import ThreadPoolExecutor
import csv as _csv
def HALT(m): raise SystemExit(f"\n{'='*70}\n[HALT] {m}\n{'='*70}")
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"]=(userdata.get("HF_TOKEN") or "").strip()
except Exception: pass
from google.colab import drive; drive.mount("/content/drive", force_remount=False)
DRIVE="/content/drive/MyDrive"
if not os.path.isdir(DRIVE): raise SystemExit("Drive not mounted")

@dataclass
class CFG:
    STAGES: tuple = ("E0","F0","F1","F2","F3","E60")  # NOT hashed (07 excludes STAGES)
    BASE: str = "inv_channel/E_INV_P0_v3"
    OUT:  str = "inv_channel/E_TRACKB2_FLUX"
    MODEL: str = "black-forest-labs/FLUX.1-dev"
    CROP: int = 1024
    SEEDS: tuple = (0,1,2)                        # 07's adapters: READ-ONLY here; kept so CFG_SHA == 07's
    ARMS: tuple = ("A","B")
    RANK: int = 16
    TARGETS: tuple = ("to_q","to_k","to_v","to_out.0")
    LR: float = 1e-4; STEPS: int = 2000; BATCH: int = 1; GRAD_ACC: int = 4
    CAPTION: str = "a photograph, sks style"
    GUIDANCE_TRAIN: float = 1.0                   # FLUX-dev is guidance-distilled
    G_PER: int = 500; GEN_SEED_BASE: int = 770000
    GEN_STEPS: int = 28; GEN_GUIDANCE: float = 3.5; GEN_BATCH: int = 2
    N_PERM: int = 10000; ALPHA: float = 0.01
    GRAD_CKPT: bool = False                       # 96 GB — off unless OOM
C=CFG(); MEAS=C.CROP

# ---- protocol hash: the SAME formula over the SAME fields as notebook 07 (gate E0 proves it) ----
def proto_sha(c):
    return hashlib.sha256(json.dumps({k:v for k,v in asdict(c).items() if k!="STAGES"},
                                     sort_keys=True,default=str).encode()).hexdigest()
CFG_SHA=proto_sha(C)

# ---- the extension: deliberately OUTSIDE CFG so that CFG_SHA stays 07's ----
NEW_SEEDS=(3,4,5)
PRESPEC="RESULTS.md Entry 60 (2026-09-15)"
EXT_SHA=hashlib.sha256(json.dumps({"v1_config_sha":CFG_SHA,"new_seeds":list(NEW_SEEDS),
                                   "prespec":PRESPEC},sort_keys=True).encode()).hexdigest()
if set(NEW_SEEDS)&set(C.SEEDS): HALT("NEW_SEEDS overlap notebook 07's seeds")
GPU_STAGES={"E0","F0","F1","F2","F3"}
if globals().get("ANALYSIS_ONLY_NO_GPU") and GPU_STAGES&set(C.STAGES):
    HALT(f"ANALYSIS_ONLY_NO_GPU=True but C.STAGES={C.STAGES} contains GPU stages; set STAGES=('E60',)")
if not globals().get("ANALYSIS_ONLY_NO_GPU") and "MIN_FREE_GIB" not in globals():
    HALT("run the GPU gate cell (first code cell) before this one")

BASE=os.path.join(DRIVE,C.BASE); ROOT=os.path.join(DRIVE,C.OUT)
if not os.path.isdir(BASE): HALT(f"base study root not found: {BASE}")
if not os.path.isdir(os.path.join(ROOT,"adapters")): HALT(f"notebook 07's output root not found: {ROOT}")
for d in ["","adapters","gens","csv","meta","train_png","figs"]: os.makedirs(os.path.join(ROOT,d),exist_ok=True)
CACHE="/content/f_cache"; os.makedirs(CACHE,exist_ok=True)
def stage(n):
    if n in C.STAGES: return True
    print(f"[{n}] SKIPPED"); return False

# pip: notebook 07's line, unchanged (unpinned; resolved versions are recorded in meta/)
subprocess.run([sys.executable,"-m","pip","install","-q","PyWavelets","diffusers>=0.31",
                "transformers","peft","accelerate","safetensors","sentencepiece","protobuf"],check=False)
import numpy as np, pandas as pd, pywt, torch
import torch.nn.functional as tF
from PIL import Image
from scipy import stats as sps
# peft/torchao: peft's is_torchao_available() RAISES on an old torchao instead of returning
# False, which kills add_adapter(). Nothing here is quantised. (Patch copied from 07.)
try:
    import peft.import_utils as _piu; _piu.is_torchao_available=lambda: False
    import peft.tuners.lora.torchao as _plt; _plt.is_torchao_available=lambda: False
    print("[ENV] peft torchao dispatcher disabled")
except Exception: pass
Image.MAX_IMAGE_PIXELS=None
DEV="cuda" if torch.cuda.is_available() else "cpu"
DT=torch.bfloat16 if DEV=="cuda" and torch.cuda.is_bf16_supported() else torch.float32
torch.backends.cudnn.allow_tf32=False; torch.backends.cuda.matmul.allow_tf32=False
np.seterr(divide="raise",invalid="raise"); _MD=DEV
_need_gpu=bool(GPU_STAGES&set(C.STAGES))
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login; login(token=os.environ["HF_TOKEN"])
elif _need_gpu: HALT("HF_TOKEN missing — FLUX.1-dev is GATED. Add it as a Colab Secret (key icon) "
                     "with Notebook access, after accepting the FLUX.1-dev licence on Hugging Face.")
if DT==torch.float16: HALT("fp16 fallback — bf16 GPU required")
if _need_gpu and DT!=torch.bfloat16: HALT(f"dtype {DT}: notebook 07 trained in bf16; a fallback breaks the recipe")
if DEV=="cuda":
    vram=torch.cuda.get_device_properties(0).total_memory/2**30
    print(f"[ENV] {torch.cuda.get_device_name(0)}  {vram:.0f} GB  dtype {DT}")
    if _need_gpu and vram<90: HALT(f"{vram:.0f} GB GPU — this notebook must not run below 90 GB")
def append_row(p,h,r):
    new=not os.path.exists(p)
    with open(p,"a") as f:
        if new: f.write(",".join(h)+"\n")
        f.write(",".join(str(x) for x in r)+"\n")
def done_keys(p,cols):
    if not os.path.exists(p): return set()
    with open(p) as f: return {tuple(r[k] for k in cols) for r in _csv.DictReader(f)}
def ajson(o,p,**kw):
    t=p+".tmp"; json.dump(o,open(t,"w"),**kw); os.replace(t,p)
def adir(t): return os.path.join(ROOT,"adapters",t)
def gdir(t): return os.path.join(ROOT,"gens",t)
V1_ARMS=[(f"{a}_raw_s{s}_flux",a,s) for s in C.SEEDS for a in C.ARMS]   # notebook 07's six (read-only)
V1_TAGS=[t for t,_,_ in V1_ARMS]
def all_arms(): return [(f"{a}_raw_s{s}_flux",a,s) for s in NEW_SEEDS for a in C.ARMS]  # the six NEW ones
def all_tags(): return [t for t,_,_ in all_arms()]
def new_only(t):
    """Every write goes through here: refuses notebook 07's tags outright."""
    if t in V1_TAGS or t not in all_tags():
        HALT(f"refusing to write for {t}: seeds {C.SEEDS} belong to notebook 07 and are read-only")
E0_JSON=os.path.join(ROOT,"meta","flux_ext_E0.json")
SNAP_JSON=os.path.join(ROOT,"meta","flux_ext_v1_snapshot.json")
EXT_CSV=os.path.join(ROOT,"csv","f3_measure_ext.csv")
def require_e0():
    if not os.path.exists(E0_JSON): HALT("stage E0 (gates) has not passed yet — run it first")
    g=json.load(open(E0_JSON))
    if not (g.get("pass") and g.get("config_sha")==CFG_SHA and g.get("ext_sha")==EXT_SHA):
        HALT(f"E0 record {E0_JSON} does not match this configuration — rerun E0")
def env_info():
    from importlib import metadata as _md
    v={}
    for p in ("torch","diffusers","transformers","peft","accelerate","safetensors","numpy","scipy","PyWavelets"):
        try: v[p]=_md.version(p)
        except Exception: v[p]=None
    v.update(python=platform.python_version(), cuda=torch.version.cuda,
             gpu=torch.cuda.get_device_name(0) if DEV=="cuda" else None)
    return v
ENV=env_info()
print("[ENV]",json.dumps(ENV))
print(f"protocol SHA {CFG_SHA[:16]} (07's) | ext SHA {EXT_SHA[:16]} | new adapters {all_tags()} | MEAS {MEAS}")


In [ ]:
# %% forensic core — byte-identical to the base study, and the base fingerprints reused
def _cp(fp): return os.path.join(CACHE,hashlib.sha1(f"{fp}|{MEAS}".encode()).hexdigest()[:16]+".npy")
def load_lum(fp,cache=True):
    if cache and os.path.exists(_cp(fp)): return np.load(_cp(fp))
    with Image.open(fp) as im:
        im=im.convert("RGB"); W,H=im.size
        a=np.asarray(im.crop(((W-MEAS)//2,(H-MEAS)//2,(W-MEAS)//2+MEAS,(H-MEAS)//2+MEAS)),np.float32)
    l=(0.299*a[...,0]+0.587*a[...,1]+0.114*a[...,2]).astype(np.float32)
    if cache: np.save(_cp(fp),l)
    return l
def load_rgb(fp):
    with Image.open(fp) as im:
        im=im.convert("RGB"); W,H=im.size
        return np.asarray(im.crop(((W-MEAS)//2,(H-MEAS)//2,(W-MEAS)//2+MEAS,(H-MEAS)//2+MEAS)),np.float32)
def _c2(x,k):
    xt=torch.from_numpy(np.ascontiguousarray(x))[None,None]; kt=torch.from_numpy(k)[None,None]
    if _MD=="cuda": return tF.conv2d(xt.cuda(),kt.cuda(),padding=k.shape[0]//2)[0,0].cpu().numpy()
    return tF.conv2d(xt,kt,padding=k.shape[0]//2)[0,0].numpy()
def wres(img):
    s0=np.float32(4.0)
    co=pywt.wavedec2(img,"db8",level=4,mode="periodization"); out=[co[0]*0.0]
    for (cH,cV,cD) in co[1:]:
        bnd=[]
        for ch in (cH,cV,cD):
            ch=ch.astype(np.float32); vm=None
            for w in (3,5,7,9):
                v=np.maximum(_c2(ch*ch,np.ones((w,w),np.float32)/(w*w))-s0,0.0)
                vm=v if vm is None else np.minimum(vm,v)
            bnd.append(ch*(s0/(vm+s0)))
        out.append(tuple(bnd))
    n=pywt.waverec2(out,"db8",mode="periodization")[:img.shape[0],:img.shape[1]]
    r=n-n.mean(); sd=float(r.std())
    if not np.isfinite(sd) or sd==0: HALT("degenerate residual")
    return (r/sd).astype(np.float32)
def nccb(a,B):
    a=a-a.mean(); na=a.norm()
    Bc=B-B.mean(dim=(1,2),keepdim=True); nb=Bc.flatten(1).norm(dim=1)
    return (Bc*a).flatten(1).sum(dim=1)/(na*nb).clamp_min(1e-30)
MAN={}
for r in _csv.DictReader(open(os.path.join(BASE,"manifest","manifest.csv"))):
    MAN.setdefault((r["role"],r["split"]),[]).append(r["path"])
KFP={r:np.load(os.path.join(BASE,"fingerprints",f"K_{r}_E2.npy")) for r in ("A","B")}
print("base manifest + fingerprints reused; T split:",len(MAN[("A","T")]),len(MAN[("B","T")]))


# training PNGs: 07 materialised them into ROOT/train_png/{A,B}. Reuse them read-only.
# Complete -> reuse. Absent/empty -> materialise with 07's exact code (writes new files only).
# Partial -> HALT (never overwrite anything 07 produced).
def tdir(role): return os.path.join(ROOT,"train_png",role)
TRAIN_SHA={}
for role in C.ARMS:
    d=tdir(role); os.makedirs(d,exist_ok=True)
    have=len(glob.glob(os.path.join(d,"*.png"))); need=len(MAN[(role,"T")])
    if have==0:
        for i,fp in enumerate(MAN[(role,"T")]):
            Image.fromarray(load_rgb(fp).astype(np.uint8)).save(
                os.path.join(d,f"{i:04d}.png"),compress_level=1)
        print(f"materialised {role}: {len(MAN[(role,'T')])} PNGs @ {MEAS}")
    elif have!=need:
        HALT(f"train_png/{role} holds {have} PNGs, manifest T split has {need}. Not touching it.")
    h=hashlib.sha256()
    for f in sorted(glob.glob(os.path.join(d,"*.png"))):
        with open(f,"rb") as fh: h.update(fh.read())
    TRAIN_SHA[role]=h.hexdigest()
    print(f"train_png/{role}: {need} PNGs, sha256 {TRAIN_SHA[role][:16]}")


In [ ]:
# %% FLUX helpers — the four places a hand-written loop goes wrong
from diffusers import FluxPipeline

def flux_load(with_text=True):
    kw=dict(torch_dtype=DT, token=True)
    if not with_text:
        kw.update(text_encoder=None, text_encoder_2=None, tokenizer=None, tokenizer_2=None)
    return FluxPipeline.from_pretrained(C.MODEL, **kw).to(DEV)

def cached_prompt():
    p=os.path.join(ROOT,"adapters","_prompt.pt")
    if os.path.exists(p):
        z=torch.load(p,map_location=DEV)
        return z["pe"].to(DEV), z["pp"].to(DEV), z["ti"].to(DEV)
    pipe=flux_load(True)
    with torch.no_grad():
        pe,pp,ti=pipe.encode_prompt(prompt=C.CAPTION,prompt_2=C.CAPTION,
                                    device=DEV,num_images_per_prompt=1,max_sequence_length=512)
    torch.save({"pe":pe.cpu(),"pp":pp.cpu(),"ti":ti.cpu()},p)
    del pipe; gc.collect(); torch.cuda.empty_cache()
    return pe,pp,ti

def calc_shift(seq_len, base_seq=256, max_seq=4096, base_shift=0.5, max_shift=1.15):
    m=(max_shift-base_shift)/(max_seq-base_seq); b=base_shift-m*base_seq
    return seq_len*m+b

def build_sigmas(sched, seq_len):
    """FLUX uses DYNAMIC shifting: the sigma schedule depends on sequence length."""
    s=copy.deepcopy(sched)
    n=int(s.config.num_train_timesteps)
    sig=np.linspace(1.0,1/n,n)
    if getattr(s.config,"use_dynamic_shifting",False):
        mu=calc_shift(seq_len)
        sig=math.exp(mu)/(math.exp(mu)+(1/sig-1)**1.0)
    else:
        sh=float(getattr(s.config,"shift",3.0)); sig=sh*sig/(1+(sh-1)*sig)
    return torch.from_numpy(np.asarray(sig,dtype=np.float32)).to(DEV)

def pack(x):
    """(B,C,H,W) -> (B, H/2*W/2, C*4)"""
    B,Cc,H,W=x.shape
    x=x.view(B,Cc,H//2,2,W//2,2).permute(0,2,4,1,3,5).reshape(B,(H//2)*(W//2),Cc*4)
    return x
def unpack(x,C_,H,W):
    """(B, H/2*W/2, C*4) -> (B,C,H,W)"""
    B=x.shape[0]
    x=x.view(B,H//2,W//2,C_,2,2).permute(0,3,1,4,2,5).reshape(B,C_,H,W)
    return x
def img_ids(H,W):
    ids=torch.zeros(H//2,W//2,3,device=DEV,dtype=DT)
    ids[...,1]=torch.arange(H//2,device=DEV)[:,None]
    ids[...,2]=torch.arange(W//2,device=DEV)[None,:]
    return ids.reshape(-1,3)
print("FLUX helpers ready: pack/unpack, dynamic-shift sigmas, image ids, cached prompt")

In [ ]:
# %% E0 — GATES. Nothing is trained until all four pass. Writes meta/flux_ext_E0.json.
SEEDBANK_ACCEPT_DRIFT=False   # set True ONLY after inspecting meta/e0_regen_*.png yourself (see below)

def _sha_file(p,chunk=1<<22):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda: f.read(chunk),b""): h.update(b)
    return h.hexdigest()
def v1_expected(t):
    _,role,seed=next(a for a in V1_ARMS if a[0]==t)
    return {"tag":t,"role":role,"seed":seed,"rank":C.RANK,"steps":C.STEPS,"meas":MEAS,
            "config_sha":CFG_SHA,"model":C.MODEL,"dtype":str(DT)}      # 07's _exp() keys
def v1_snapshot():
    s={"adapters":{},"gens":{},"files":{}}
    for t in V1_TAGS:
        s["adapters"][t]={"weights_sha256":_sha_file(os.path.join(adir(t),"pytorch_lora_weights.safetensors")),
                          "meta_sha256":_sha_file(os.path.join(adir(t),"train_meta.json"))}
        g=glob.glob(os.path.join(gdir(t),"*.png"))
        s["gens"][t]={"n_png":len(g),"bytes":int(sum(os.path.getsize(x) for x in g))}
    for f in ("csv/f3_measure.csv","E4_flux_results.json","adapters/_prompt.pt"):
        p=os.path.join(ROOT,f); s["files"][f]=_sha_file(p) if os.path.exists(p) else None
    s["train_png_sha256"]=TRAIN_SHA
    return s
def verify_snapshot():
    """Returns [] if every v1 artefact recorded at E0 is unchanged."""
    if not os.path.exists(SNAP_JSON): HALT("no v1 snapshot — run E0 first")
    old=json.load(open(SNAP_JSON)); now=v1_snapshot(); diffs=[]
    for sec in ("adapters","gens","train_png_sha256"):
        if old[sec]!=now[sec]: diffs.append(sec)
    for f,v in old["files"].items():
        if v is not None and now["files"].get(f)!=v: diffs.append(f)
    return diffs
def _lum_u8(img): a=np.asarray(img.convert("RGB"),np.float64); return 0.299*a[...,0]+0.587*a[...,1]+0.114*a[...,2]

def e0():
    rep={"config_sha":CFG_SHA,"ext_sha":EXT_SHA,"prespec":PRESPEC,"new_seeds":list(NEW_SEEDS),
         "env":ENV,"train_png_sha256":TRAIN_SHA,"time":time.strftime("%Y-%m-%d %H:%M:%S")}
    # (i) protocol hash against what 07 stored -------------------------------------------------
    stored={}; bad=[]
    for t in V1_TAGS:
        pm=os.path.join(adir(t),"train_meta.json")
        if not (os.path.exists(pm) and os.path.exists(os.path.join(adir(t),"pytorch_lora_weights.safetensors"))):
            HALT(f"notebook 07 adapter missing or incomplete: {adir(t)}")
        m=json.load(open(pm)); stored[t]=m.get("config_sha")
        diff={k:[m.get(k),v] for k,v in v1_expected(t).items() if m.get(k)!=v}
        if diff or not m.get("effective",False): bad.append((t,diff))
    p4=os.path.join(ROOT,"E4_flux_results.json")
    e4_sha=json.load(open(p4)).get("config_sha") if os.path.exists(p4) else None
    if e4_sha is not None and e4_sha!=CFG_SHA: bad.append(("E4_flux_results.json",{"config_sha":[e4_sha,CFG_SHA]}))
    if bad:
        # two execution-only fields also sit inside 07's hash; name them if they explain it
        hits=[(g_,b_) for g_ in (False,True) for b_ in (1,2,3,4,6,8)
              if proto_sha(replace(C,GRAD_CKPT=g_,GEN_BATCH=b_)) in set(stored.values())]
        for t,d_ in bad: print(f"  MISMATCH {t}: {d_}")
        HALT("protocol hash does not reproduce notebook 07's stored config_sha.\n" +
             (f"It matches with GRAD_CKPT/GEN_BATCH = {hits} — set those in the CFG cell to what 07 "
              "ran with and rerun." if hits else
              "No GRAD_CKPT/GEN_BATCH variant explains it: the 07 run used a configuration other "
              "than the one in the repo. Do not proceed; the new adapters would not be comparable."))
    rep["protocol_hash"]={"pass":True,"stored":stored,"e4_results_sha":e4_sha}
    print(f"[E0-i] PASS — CFG_SHA {CFG_SHA[:16]} equals the config_sha of all six 07 adapters"
          + (" and of E4_flux_results.json" if e4_sha else ""))
    # (ii) read-only snapshot of 07's artefacts ------------------------------------------------
    if os.path.exists(SNAP_JSON):
        diffs=verify_snapshot()
        if diffs: HALT(f"07's artefacts changed since the first E0 snapshot: {diffs}")
        print("[E0-ii] PASS — 07's artefacts unchanged since the snapshot")
    else:
        snap=v1_snapshot(); snap["time"]=rep["time"]; ajson(snap,SNAP_JSON,indent=2)
        print(f"[E0-ii] snapshot of 07's artefacts written: {SNAP_JSON}")
    rep["snapshot"]=SNAP_JSON
    # (iii) seed bank: regenerate 07's images 00000/00001 exactly as 07's F2 did (batch of 2) -----
    ref=next((t for t in V1_TAGS if all(os.path.exists(os.path.join(gdir(t),f"{i:05d}.png")) for i in range(4))),None)
    if ref is None: HALT("no notebook 07 adapter has stored generations 00000-00003")
    ix=list(range(C.GEN_BATCH)) if C.GEN_BATCH>=2 else [0,1]
    pipe=flux_load(True); pipe.set_progress_bar_config(disable=True)
    try:
        pipe.load_lora_weights(adir(ref))
        gs=[torch.Generator(device="cpu").manual_seed(C.GEN_SEED_BASE+i) for i in ix]
        with torch.inference_mode():
            imgs=pipe(prompt=[C.CAPTION]*len(ix),height=MEAS,width=MEAS,
                      num_inference_steps=C.GEN_STEPS,guidance_scale=C.GEN_GUIDANCE,
                      generator=gs,max_sequence_length=512).images
    finally:
        del pipe; gc.collect(); torch.cuda.empty_cache()
    sb={"reference":ref,"images":{}}
    for i,im in zip(ix[:2],imgs[:2]):
        im.save(os.path.join(ROOT,"meta",f"e0_regen_{ref}_{i:05d}.png"),format="PNG")
        new=_lum_u8(im); same=_lum_u8(Image.open(os.path.join(gdir(ref),f"{i:05d}.png")))
        other=_lum_u8(Image.open(os.path.join(gdir(ref),f"{1-i:05d}.png")))
        ident=bool((np.asarray(im.convert("RGB"))==np.asarray(Image.open(os.path.join(gdir(ref),f"{i:05d}.png")).convert("RGB"))).all())
        sb["images"][i]={"identical":ident,"mean_abs_diff_255":float(np.abs(new-same).mean()),
                         "r_same":float(np.corrcoef(new.ravel(),same.ravel())[0,1]),
                         "r_other_seed":float(np.corrcoef(new.ravel(),other.ravel())[0,1])}
        print(f"[E0-iii] {ref} image {i:05d}: identical {ident} | mean|diff| "
              f"{sb['images'][i]['mean_abs_diff_255']:.3f}/255 | r(same seed) {sb['images'][i]['r_same']:.4f} "
              f"| r(other seed) {sb['images'][i]['r_other_seed']:.4f}")
    allid=all(v["identical"] for v in sb["images"].values())
    near=all(v["r_same"]>=0.90 and v["r_same"]-v["r_other_seed"]>=0.20 for v in sb["images"].values())
    if allid: sb["verdict"]="identical"
    elif near: sb["verdict"]="near (numerical drift across GPU/library versions)"
    elif SEEDBANK_ACCEPT_DRIFT: sb["verdict"]="accepted by user after inspection"
    else:
        HALT("seed bank check failed: the regenerated image is not the stored one "
             "(r(same seed) < 0.90 or not clearly closer than a different seed).\nCompare "
             f"{ROOT}/meta/e0_regen_*.png with {gdir(ref)}/00000.png. Same composition, only fine "
             "detail differs -> numerical drift:\nset SEEDBANK_ACCEPT_DRIFT=True and rerun E0. "
             "Different picture -> the seed bank or caption is wrong: do not train.")
    rep["seed_bank"]=sb
    print(f"[E0-iii] PASS — seed bank: {sb['verdict']}")
    if not allid:
        print("         Not bit-identical: the new adapters are generated in a different numerical "
              "environment\n         (GPU architecture / library versions). Both arms share it, so the "
              "paired own-minus-other\n         contrast is unaffected; the difference is recorded and "
              "must be disclosed.")
    # (iv) measurement: this notebook's forensic core vs 07's stored rows -------------------------
    p3=os.path.join(ROOT,"csv","f3_measure.csv")
    if not os.path.exists(p3): HALT(f"07's measurement CSV missing: {p3}")
    d=pd.read_csv(p3,dtype={"tag":str,"K":str})
    n3=len(d); exp3=len(V1_TAGS)*C.G_PER*2
    if n3!=exp3: HALT(f"07's f3_measure.csv has {n3} rows, expected {exp3}")
    K=torch.stack([torch.from_numpy(KFP[r]) for r in ("A","B")]).to(_MD); mx=0.0; rows=[]
    for i in range(4):
        Z_np=load_lum(os.path.join(gdir(ref),f"{i:05d}.png"),False)
        W=torch.from_numpy(wres(Z_np)).to(_MD); Z=torch.from_numpy(Z_np).to(_MD)
        v=nccb(W,Z.unsqueeze(0)*K).cpu().numpy()
        for j,r in enumerate(("A","B")):
            old=float(d[(d.tag==ref)&(d.gen_idx==i)&(d.K==r)]["rho"].iloc[0])
            rel=abs(float(v[j])-old)/max(abs(old),1e-12); mx=max(mx,rel)
            rows.append({"gen_idx":i,"K":r,"stored":old,"now":float(v[j]),"rel_diff":rel})
    if mx>1e-4: HALT(f"measurement does not reproduce 07's rows (max relative diff {mx:.2e}); see {rows}")
    rep["measurement"]={"reference":ref,"max_rel_diff":mx,"rows":rows,"v1_rows":n3}
    print(f"[E0-iv] PASS — forensic core reproduces 07's rho on 8 stored values (max rel diff {mx:.1e})")
    rep["pass"]=True; ajson(rep,E0_JSON,indent=2)
    print(f"\n[E0] PASS — all gates. Record: {E0_JSON}")
if stage("E0"): e0()


In [ ]:
# %% F0 — 5-step SMOKE TEST. Surfaces shape/dtype errors in <1 min, not 30.
def smoke():
    from peft import LoraConfig
    pe,pp,ti=cached_prompt()
    pipe=flux_load(False)
    vae,tr,sch=pipe.vae,pipe.transformer,pipe.scheduler
    vae.requires_grad_(False); tr.requires_grad_(False)
    tr.add_adapter(LoraConfig(r=C.RANK,lora_alpha=C.RANK,init_lora_weights="gaussian",
                              target_modules=list(C.TARGETS)))
    if C.GRAD_CKPT: tr.enable_gradient_checkpointing()
    tr.train()
    ps=[p for p in tr.parameters() if p.requires_grad]
    print(f"[F0] trainable LoRA params: {sum(p.numel() for p in ps):,}")
    sf=float(vae.config.scaling_factor); sh=float(vae.config.shift_factor or 0.0)
    f=sorted(glob.glob(os.path.join(tdir("A"),"*.png")))[:4]
    px=torch.stack([torch.from_numpy(
        np.asarray(Image.open(x).convert("RGB"),np.float32)/127.5-1.0).permute(2,0,1) for x in f])
    opt=torch.optim.AdamW(ps,lr=C.LR)
    for st in range(5):
        b=px[st%len(f):st%len(f)+C.BATCH].to(DEV,dtype=DT)
        with torch.no_grad():
            lat=vae.encode(b).latent_dist.sample(); lat=((lat-sh)*sf).to(DT)
        B_,Cl,Hl,Wl=lat.shape
        seq=(Hl//2)*(Wl//2)
        if st==0:
            print(f"[F0] latent {tuple(lat.shape)}  seq_len {seq}  "
                  f"packed {(B_,seq,Cl*4)}")
            assert Cl==16, f"expected 16 latent channels, got {Cl}"
        sig_all=build_sigmas(sch,seq); n=len(sig_all)
        u=torch.sigmoid(torch.randn(B_,device=DEV,dtype=torch.float32))
        ii=(u*n).long().clamp_(0,n-1)
        sg=sig_all[ii].to(lat.dtype).view(-1,1,1,1)
        noise=torch.randn_like(lat)
        noisy=(1.0-sg)*lat+sg*noise
        pk=pack(noisy)
        ids=img_ids(Hl,Wl)
        guid=torch.full((B_,),C.GUIDANCE_TRAIN,device=DEV,dtype=DT)
        out=tr(hidden_states=pk, timestep=(sig_all[ii]).to(DT),
               guidance=guid, pooled_projections=pp.repeat(B_,1).to(DT),
               encoder_hidden_states=pe.repeat(B_,1,1).to(DT),
               txt_ids=ti.to(DT), img_ids=ids, return_dict=False)[0]
        if st==0: print(f"[F0] transformer out {tuple(out.shape)}")
        pred=unpack(out,Cl,Hl,Wl)
        loss=tF.mse_loss(pred.float(),(noise-lat).float())
        loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
        print(f"[F0] step {st+1}/5 loss {float(loss):.5f} "
              f"peak {torch.cuda.max_memory_allocated()/2**30:.1f} GB")
        if not np.isfinite(float(loss)): HALT("non-finite loss in smoke test")
    print(f"\n[F0] PASS — shapes, dtypes and gradients all sound. "
          f"Peak {torch.cuda.max_memory_allocated()/2**30:.1f} GB of "
          f"{torch.cuda.get_device_properties(0).total_memory/2**30:.0f} GB.")
    del pipe,tr,vae; gc.collect(); torch.cuda.empty_cache()
if stage("F0"): smoke()

In [ ]:
# %% F1 — train the 6 NEW adapters (seeds 3-5 x 2 arms). Code copied from 07's F1;
#     edits: write guard, E0 prerequisite, ext_sha + environment in train_meta.json.
def _exp(t):
    _,role,seed=next(a for a in all_arms() if a[0]==t)
    return {"tag":t,"role":role,"seed":seed,"rank":C.RANK,"steps":C.STEPS,"meas":MEAS,
            "config_sha":CFG_SHA,"model":C.MODEL,"dtype":str(DT),
            "ext_sha":EXT_SHA}
def avalid(t):
    p=os.path.join(adir(t),"train_meta.json")
    if not os.path.exists(os.path.join(adir(t),"pytorch_lora_weights.safetensors")): return False
    m=json.load(open(p)) if os.path.exists(p) else None
    return m is not None and all(m.get(k)==v for k,v in _exp(t).items()) and m.get("effective",False)

def train_one(tag,pe,pp,ti):
    _,role,seed=next(a for a in all_arms() if a[0]==tag)
    if avalid(tag): print(f"[F1] skip {tag}"); return
    new_only(tag)
    from peft import LoraConfig
    from peft.utils import get_peft_model_state_dict
    out=adir(tag)
    if os.path.exists(out): shutil.move(out,out+f"__stale_{int(time.time())}")
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    if DEV=="cuda": torch.cuda.reset_peak_memory_stats()
    pipe=flux_load(False); vae,tr,sch=pipe.vae,pipe.transformer,pipe.scheduler
    vae.requires_grad_(False); tr.requires_grad_(False)
    tr.add_adapter(LoraConfig(r=C.RANK,lora_alpha=C.RANK,init_lora_weights="gaussian",
                              target_modules=list(C.TARGETS)))
    if C.GRAD_CKPT: tr.enable_gradient_checkpointing()
    tr.train()
    ps=[p for p in tr.parameters() if p.requires_grad]
    opt=torch.optim.AdamW(ps,lr=C.LR,weight_decay=1e-4)
    lrs=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=math.ceil(C.STEPS/C.GRAD_ACC))
    opt.zero_grad(set_to_none=True)
    files=sorted(glob.glob(os.path.join(tdir(role),"*.png")))
    cache=torch.stack([torch.from_numpy(
        np.asarray(Image.open(f).convert("RGB"),np.float32).copy()/127.5-1.0).permute(2,0,1)
        for f in files])
    sf=float(vae.config.scaling_factor); shf=float(vae.config.shift_factor or 0.0)
    g=torch.Generator().manual_seed(seed); losses=[]; t0=time.time(); sig_all=None; ids=None
    print(f"\n[F1] {tag}: role={role} seed={seed}, {C.STEPS} steps")
    for st in range(C.STEPS):
        idx=torch.randint(0,cache.shape[0],(C.BATCH,),generator=g)
        px=cache[idx].to(DEV,dtype=DT)
        with torch.no_grad():
            lat=vae.encode(px).latent_dist.sample(); lat=((lat-shf)*sf).to(DT)
        B_,Cl,Hl,Wl=lat.shape
        if sig_all is None:
            sig_all=build_sigmas(sch,(Hl//2)*(Wl//2)); ids=img_ids(Hl,Wl)
        n=len(sig_all)
        u=torch.sigmoid(torch.randn(B_,device=DEV,dtype=torch.float32))
        ii=(u*n).long().clamp_(0,n-1); sg=sig_all[ii].to(lat.dtype).view(-1,1,1,1)
        noise=torch.randn_like(lat)
        pk=pack((1.0-sg)*lat+sg*noise)
        guid=torch.full((B_,),C.GUIDANCE_TRAIN,device=DEV,dtype=DT)
        outp=tr(hidden_states=pk,timestep=sig_all[ii].to(DT),guidance=guid,
                pooled_projections=pp.repeat(B_,1).to(DT),
                encoder_hidden_states=pe.repeat(B_,1,1).to(DT),
                txt_ids=ti.to(DT),img_ids=ids,return_dict=False)[0]
        pred=unpack(outp,Cl,Hl,Wl)
        loss=tF.mse_loss(pred.float(),(noise-lat).float())/C.GRAD_ACC
        loss.backward()
        if (st+1)%C.GRAD_ACC==0:
            torch.nn.utils.clip_grad_norm_(ps,1.0); opt.step(); lrs.step()
            opt.zero_grad(set_to_none=True)
        losses.append(float(loss.detach())*C.GRAD_ACC)
        if not np.isfinite(losses[-1]): HALT(f"{tag}: non-finite loss at step {st+1}")
        if (st+1)%200==0:
            el=time.time()-t0
            print(f"    {st+1}/{C.STEPS} loss {np.mean(losses[-200:]):.5f} "
                  f"eta {(el/(st+1))*(C.STEPS-st-1)/60:.1f}min "
                  f"peak {torch.cuda.max_memory_allocated()/2**30:.0f}GB")
    sd=get_peft_model_state_dict(tr)
    bn=float(sum(float(v.float().norm()) for k,v in sd.items() if "lora_B" in k))
    if not (bn>1e-6 and all(bool(torch.isfinite(v).all()) for v in sd.values())):
        HALT(f"{tag}: DEAD adapter (||lora_B||={bn:.3e})")
    os.makedirs(out,exist_ok=True)
    FluxPipeline.save_lora_weights(save_directory=out,transformer_lora_layers=sd)
    m=_exp(tag); m.update({"loss_tail":float(np.mean(losses[-100:])),"lora_B_norm":bn,
                           "effective":True,"minutes":(time.time()-t0)/60,
                           "peak_gb":float(torch.cuda.max_memory_allocated()/2**30),
                           "prespec":PRESPEC,"train_png_sha256":TRAIN_SHA[role],"env":ENV})
    ajson(m,os.path.join(out,"train_meta.json"),indent=2)
    print(f"[F1] saved {tag}: ||lora_B||={bn:.2f} tail={m['loss_tail']:.5f} "
          f"({m['minutes']:.1f} min)")
    del pipe,tr,vae,cache; gc.collect(); torch.cuda.empty_cache()

if stage("F1"):
    require_e0()
    pe,pp,ti=cached_prompt()
    for i,t in enumerate(all_tags(),1):
        print(f"[F1] adapter {i}/{len(all_tags())}: {t}"); train_one(t,pe,pp,ti)
    bad=[t for t in all_tags() if not avalid(t)]
    if bad: HALT(f"F1 incomplete: {bad}")
    print(f"[F1] PASS — {len(all_tags())} adapters")

In [ ]:
# %% F2 — generation for the NEW adapters, same paired seed bank (copied from 07's F2;
#     edits: write guard, E0 prerequisite, adapter provenance file per gens dir)
def f2():
    if DEV!="cuda": HALT("F2 needs a GPU")
    require_e0()
    bad=[t for t in all_tags() if not avalid(t)]
    if bad: HALT(f"F2 needs every new adapter trained and valid first: {bad}")
    bank=[(C.CAPTION,C.GEN_SEED_BASE+j) for j in range(C.G_PER)]
    pipe=flux_load(True); pipe.set_progress_bar_config(disable=True); prev=False
    try:
        for tag in all_tags():
            new_only(tag)
            d_=gdir(tag); os.makedirs(d_,exist_ok=True)
            src={"weights_sha256":hashlib.sha256(open(os.path.join(adir(tag),"pytorch_lora_weights.safetensors"),"rb").read()).hexdigest(),"ext_sha":EXT_SHA}
            sp=os.path.join(d_,"_source.json")
            if os.path.exists(sp) and json.load(open(sp))!=src:
                HALT(f"{d_} holds generations from a different adapter; move it aside first")
            if not os.path.exists(sp): ajson(src,sp,indent=2)
            for t_ in glob.glob(os.path.join(d_,"*.tmp")): os.remove(t_)
            miss=[i for i in range(C.G_PER) if not os.path.exists(os.path.join(d_,f"{i:05d}.png"))]
            if not miss: print(f"[F2] skip {tag}"); continue
            if prev: pipe.unload_lora_weights(); prev=False
            pipe.load_lora_weights(adir(tag)); prev=True
            print(f"[F2] {tag}: {len(miss)} missing")
            t0=time.time()
            for s in range(0,len(miss),C.GEN_BATCH):
                ix=miss[s:s+C.GEN_BATCH]
                gs=[torch.Generator(device="cpu").manual_seed(bank[i][1]) for i in ix]
                with torch.inference_mode():
                    imgs=pipe(prompt=[bank[i][0] for i in ix],height=MEAS,width=MEAS,
                              num_inference_steps=C.GEN_STEPS,guidance_scale=C.GEN_GUIDANCE,
                              generator=gs,max_sequence_length=512).images
                for i,im in zip(ix,imgs):
                    t_=os.path.join(d_,f"{i:05d}.png.tmp")
                    im.save(t_,format="PNG"); os.replace(t_,os.path.join(d_,f"{i:05d}.png"))
                if (s//C.GEN_BATCH)%25==0:
                    el=time.time()-t0
                    print(f"    {s+len(ix)}/{len(miss)}  eta "
                          f"{(el/max(s+len(ix),1))*(len(miss)-s-len(ix))/60:.0f}min")
            print(f"[F2] complete {tag} ({(time.time()-t0)/60:.0f} min)")
    finally:
        del pipe; gc.collect(); torch.cuda.empty_cache()
    print("[F2] PASS")
if stage("F2"): f2()

In [ ]:
# %% F3 — measurement of the NEW generations against the BASE STUDY fingerprints (identical
#     statistic; copied from 07's F3). Writes csv/f3_measure_ext.csv; 07's CSV is never touched.
def f3():
    K=torch.stack([torch.from_numpy(KFP[r]) for r in ("A","B")]).to(_MD)
    p=EXT_CSV; done=done_keys(p,["tag","gen_idx","K"])
    ex=ThreadPoolExecutor(max_workers=4)
    try:
        for tag in all_tags():
            new_only(tag)
            paths=[os.path.join(gdir(tag),f"{i:05d}.png") for i in range(C.G_PER)]
            todo=[i for i in range(C.G_PER) if not all((tag,str(i),r) in done for r in ("A","B"))]
            if not todo: continue
            fut={i:ex.submit(load_lum,paths[i],False) for i in todo[:6]}; t0=time.time()
            for pos,i in enumerate(todo):
                Z_np=fut.pop(i).result(); nx=pos+6
                if nx<len(todo): fut[todo[nx]]=ex.submit(load_lum,paths[todo[nx]],False)
                W=torch.from_numpy(wres(Z_np)).to(_MD); Z=torch.from_numpy(Z_np).to(_MD)
                v=nccb(W,Z.unsqueeze(0)*K).cpu().numpy()
                for j,r in enumerate(("A","B")):
                    if (tag,str(i),r) in done: continue
                    append_row(p,["tag","gen_idx","K","rho"],[tag,i,r,f"{v[j]:.6e}"])
            print(f"[F3] {tag}: {len(todo)} ({time.time()-t0:.0f}s)")
    finally:
        ex.shutdown(wait=True); gc.collect()
        if DEV=="cuda": torch.cuda.empty_cache()
    n=len(done_keys(p,["tag","gen_idx","K"])); exp=len(all_tags())*C.G_PER*2
    if n!=exp: HALT(f"F3 incomplete: {n}/{exp}")
    print(f"[F3] PASS — {n}/{exp} rows")
if stage("F3"): f3()

In [ ]:
# %% E60 — the analysis pre-specified in RESULTS.md Entry 60. Runs ONLY when all twelve adapters
#     (07's seeds 0-2 + seeds 3-5, both arms) are completely measured: no early look.
R_REAL_V1=3.56703e-02               # 07's F4 literal ("base-study device-specific denominator")
R_REAL_LEDGER=0.0356703416571125    # analysis/FINAL_LEDGER.json -> denominators.R_real (same quantity)
# 07's published per-seed contrasts (docs/E_INV_RESULTS_v2.md sec. 6.1), used if E4_flux_results.json is absent
V1_DOC={"A":[-7.949e-06,6.368e-05,-1.548e-05],"B":[-6.402e-05,9.713e-06,3.070e-05],
        "U_device":2.7717e-04,"lambda_U_pct":0.777}
SD35_K6_LAMBDA_PCT=0.249            # FINAL_LEDGER history_k6: SD-3.5 max-arm at six adapters per arm

def exact_signflip(m):
    """One-sided exact sign-flip p over all 2^k sign patterns (notebook 12's definition)."""
    obs=float(np.mean(m))
    return sum(1 for sg in itertools.product([-1,1],repeat=len(m))
               if float(np.mean(m*np.array(sg)))>=obs)/2**len(m)

def e60():
    ALPHA=C.ALPHA                                     # 0.01, one-sided
    v1=pd.read_csv(os.path.join(ROOT,"csv","f3_measure.csv"),dtype={"tag":str,"K":str})
    if not os.path.exists(EXT_CSV): HALT("no new measurements yet — run F1-F3 first")
    ex=pd.read_csv(EXT_CSV,dtype={"tag":str,"K":str})
    # ---- completeness: every adapter, every generation, both fingerprints, no duplicates ----
    for d_,tags,name in ((v1,V1_TAGS,"07"),(ex,all_tags(),"ext")):
        if d_.duplicated(["tag","gen_idx","K"]).any(): HALT(f"duplicate rows in the {name} CSV")
        for t in tags:
            for r in C.ARMS:
                g=set(d_[(d_.tag==t)&(d_.K==r)]["gen_idx"].astype(int))
                if g!=set(range(C.G_PER)):
                    HALT(f"{name}: {t} x K_{r} has {len(g)}/{C.G_PER} generations measured. Entry 60 is "
                         "read only on the complete design - finish F1-F3 first.")
    diffs=verify_snapshot()
    if diffs: HALT(f"07's artefacts changed since the E0 snapshot: {diffs}")
    def th(d,tag,ks,ko):                              # 07's F4 contrast, verbatim logic
        a=d[(d.tag==tag)&(d.K==ks)].sort_values("gen_idx")["rho"].to_numpy(float)
        b=d[(d.tag==tag)&(d.K==ko)].sort_values("gen_idx")["rho"].to_numpy(float)
        n=min(len(a),len(b)); return a[:n]-b[:n]
    per={}
    for arm,ks,ko in (("A","A","B"),("B","B","A")):
        rows=[]
        for s in sorted(C.SEEDS+NEW_SEEDS):           # integer-seed order, never lexicographic
            t=f"{arm}_raw_s{s}_flux"; x=th(v1 if s in C.SEEDS else ex,t,ks,ko)
            m=json.load(open(os.path.join(adir(t),"train_meta.json")))
            rows.append({"tag":t,"seed":s,"source":"07" if s in C.SEEDS else "07b","theta":float(x.mean()),
                         "n_gen":int(len(x)),"lora_B_norm":m.get("lora_B_norm")})
        per[arm]=rows
    # ---- gate: 07's three-adapter result must be reproduced from 07's own rows ----
    p4=os.path.join(ROOT,"E4_flux_results.json"); e4=json.load(open(p4)) if os.path.exists(p4) else None
    t2=float(sps.t.ppf(1-ALPHA/2,df=2)); Uv1=[]
    for a in C.ARMS:
        mv=np.array([r["theta"] for r in per[a] if r["source"]=="07"])
        ref=np.array(e4["arms"][a]["means"]) if e4 else np.array(V1_DOC[a])
        tol=1e-9 if e4 else 2e-4
        if not np.allclose(mv,ref,rtol=tol,atol=1e-12):
            HALT(f"07's arm {a} contrasts do not reproduce: {mv} vs {ref}")
        Uv1.append(float(mv.mean()+t2*mv.std(ddof=1)/np.sqrt(len(mv))))
    U_v1=max(Uv1)
    if abs(U_v1-V1_DOC["U_device"])>2e-4*V1_DOC["U_device"]: HALT(f"07's U_device not reproduced: {U_v1:.5e}")
    # ---- denominator: the one 07 used for FLUX ----
    R=R_REAL_V1; R_src="notebook 07 F4 literal R_real=3.56703e-02"
    if e4 and e4.get("lambda_U_n3"):
        R_e4=e4["U_device"]/e4["lambda_U_n3"]
        if abs(R_e4-R)>1e-9*R: HALT(f"E4_flux_results.json implies R_real={R_e4:.8e}, not {R:.8e}")
        R_src+="; confirmed from E4_flux_results.json (U_device/lambda_U_n3)"
    print(f"[E60] 07 reproduced: U_device(n=3) {U_v1:.4e} = {U_v1/R:.3%} (published 0.777 %)")
    print(f"[E60] R_real = {R:.6e}  ({R_src}; FINAL_LEDGER value {R_REAL_LEDGER:.10e}, "
          f"rel. diff {abs(R-R_REAL_LEDGER)/R_REAL_LEDGER:.1e})\n")
    # ---- Entry 60 statistics ----
    arms={}
    print(f"{'arm':>4}{'k':>4}{'mean':>13}{'sd':>12}{'t_crit':>8}{'U':>13}{'exact p':>10}{'floor':>9}")
    print("-"*73)
    for a in C.ARMS:
        m=np.array([r["theta"] for r in per[a]]); k=len(m)
        if k!=6: HALT(f"arm {a}: {k} adapters, Entry 60 specifies six")
        tc=float(sps.t.ppf(1-ALPHA/2,df=k-1))        # t_{0.995,5} = 4.0321
        U=float(m.mean()+tc*m.std(ddof=1)/np.sqrt(k)); p=exact_signflip(m)
        arms[a]={"adapters":per[a],"k":k,"mean":float(m.mean()),"sd":float(m.std(ddof=1)),
                 "t_crit_0995":tc,"U":U,"exact_signflip_p":p,"floor":1/2**k}
        print(f"{a:>4}{k:>4}{m.mean():>13.4e}{m.std(ddof=1):>12.3e}{tc:>8.4f}{U:>13.4e}{p:>10.4f}{1/2**k:>9.5f}")
        for r in per[a]: print(f"      {r['tag']:18s} [{r['source']:>3s}] {r['theta']:+.4e}")
    U_dev=max(arms[a]["U"] for a in C.ARMS)
    a_=np.array([r["theta"] for r in per["A"]]); b_=np.array([r["theta"] for r in per["B"]])
    th_sym=0.5*(a_.mean()+b_.mean())                  # own-minus-other in both arms
    vA=a_.var(ddof=1)/len(a_); vB=b_.var(ddof=1)/len(b_)
    se=0.5*np.sqrt(vA+vB)                             # Welch adapter-level SE (src/dose_stats.py)
    dfw=(vA+vB)**2/(vA**2/(len(a_)-1)+vB**2/(len(b_)-1))
    t_sym=th_sym/se; p_sym=float(sps.t.sf(t_sym,dfw))
    lim_sym=float(th_sym+sps.t.ppf(1-ALPHA,dfw)*se)   # one-sided 99 % limit
    # ---- readings, in Entry 60's words ----
    floor=1/2**6
    iut_level=max(ALPHA,floor)                        # attainable floor: 1/64 > 0.01 at k=6
    iut_reject=all(arms[a]["exact_signflip_p"]<=iut_level for a in C.ARMS)
    sym_reject=p_sym<ALPHA
    both_pos=arms["A"]["mean"]>0 and arms["B"]["mean"]>0
    if sym_reject and both_pos:
        code="R2"; reading="device-specific transfer on FLUX.1-dev"
    elif not iut_reject and not sym_reject:
        code="R1"; reading="no detectable transfer on FLUX.1-dev at six adapters per arm"
    else:
        code="R3"; reading="neither pre-specified reading applies: report the estimate and limits only"
    # ---- descriptive, NOT part of Entry 60: 07's adapters vs the new ones (environment differs) ----
    desc={}
    for a in C.ARMS:
        o=np.array([r["theta"] for r in per[a] if r["source"]=="07"])
        n_=np.array([r["theta"] for r in per[a] if r["source"]=="07b"])
        vo,vn=o.var(ddof=1)/len(o),n_.var(ddof=1)/len(n_)
        dfd=(vo+vn)**2/(vo**2/(len(o)-1)+vn**2/(len(n_)-1)); tt=(n_.mean()-o.mean())/np.sqrt(vo+vn)
        desc[a]={"mean_07":float(o.mean()),"mean_07b":float(n_.mean()),"welch_t":float(tt),"welch_df":float(dfd),
                 "two_sided_p":float(2*sps.t.sf(abs(tt),dfd))}
    print("\n"+"="*76)
    print(f"ENTRY 60 · FLUX.1-dev, six adapters per arm (seeds 0-5), alpha {ALPHA} one-sided")
    print("="*76)
    print(f"Intersection-union sign-flip test: p_A = {arms['A']['exact_signflip_p']:.4f}, "
          f"p_B = {arms['B']['exact_signflip_p']:.4f}  (floor 1/64 = {floor:.4f} > {ALPHA})")
    print(f"   evaluated at its attainable floor: rejects only if both p <= {iut_level:.4f} -> "
          f"{'REJECTS' if iut_reject else 'does not reject'}")
    print(f"Max-arm limit (t_0.995,5 = {arms['A']['t_crit_0995']:.4f}): U_device = {U_dev:.4e}  "
          f"->  lambda_U = {U_dev/R:.4%}")
    print(f"Symmetric: theta_sym = {th_sym:+.4e} (lambda_hat {th_sym/R:+.4%}), Welch SE {se:.3e}, "
          f"df {dfw:.2f}, t {t_sym:.2f}, one-sided p {p_sym:.4f} -> "
          f"{'REJECTS' if sym_reject else 'does not reject'} at {ALPHA}")
    print(f"   one-sided 99 % limit = {lim_sym:.4e}  ->  {lim_sym/R:.4%} of R_real")
    print(f"Arms positive: A {arms['A']['mean']>0}, B {arms['B']['mean']>0}")
    print("-"*76)
    print(f"READING [{code}]: \"{reading}\"")
    print("-"*76)
    print(f"Context: FLUX at n=3 (07) lambda_U {U_v1/R:.3%}; SD-3.5 max-arm at six per arm "
          f"{SD35_K6_LAMBDA_PCT} % (FINAL_LEDGER history_k6).")
    print("Descriptive only (not pre-specified): 07 adapters vs new adapters, trained in different "
          "environments:")
    for a in C.ARMS:
        print(f"   arm {a}: 07 mean {desc[a]['mean_07']:+.3e}, new mean {desc[a]['mean_07b']:+.3e}, "
              f"Welch t {desc[a]['welch_t']:+.2f} (df {desc[a]['welch_df']:.1f}), p {desc[a]['two_sided_p']:.3f}")
    print("="*76)
    out={"prespecification":PRESPEC,"notebook":"notebooks/07b_flux_seed_ext.ipynb",
         "model":C.MODEL,"config_sha":CFG_SHA,"ext_sha":EXT_SHA,"alpha_one_sided":ALPHA,
         "R_real":R,"R_real_source":R_src,"R_real_ledger":R_REAL_LEDGER,
         "arms":arms,
         "iut":{"p_A":arms["A"]["exact_signflip_p"],"p_B":arms["B"]["exact_signflip_p"],
                "floor":floor,"attainable_level":iut_level,"rejects":bool(iut_reject),
                "note":"at k=6 the smallest attainable p is 1/64 > 0.01; Entry 60 evaluates the IUT at that floor"},
         "max_arm":{"t_crit_0995_df5":arms["A"]["t_crit_0995"],"U_A":arms["A"]["U"],"U_B":arms["B"]["U"],
                    "U_device":U_dev,"lambda_U":U_dev/R,"lambda_U_pct":100*U_dev/R},
         "symmetric":{"theta_A":arms["A"]["mean"],"theta_B":arms["B"]["mean"],"theta_sym":float(th_sym),
                      "lambda_hat_pct":float(100*th_sym/R),"welch_se":float(se),"welch_df":float(dfw),
                      "t":float(t_sym),"p_one_sided":p_sym,"rejects":bool(sym_reject),
                      "limit_99_one_sided":lim_sym,"lambda_sym_U_pct":float(100*lim_sym/R)},
         "both_arms_positive":bool(both_pos),"reading_code":code,"reading":reading,
         "v1_reproduced":{"U_device_n3":U_v1,"lambda_U_n3_pct":100*U_v1/R,"published_pct":V1_DOC["lambda_U_pct"]},
         "context":{"sd35_maxarm_k6_lambda_pct":SD35_K6_LAMBDA_PCT},
         "descriptive_not_prespecified":{"v1_vs_ext_by_arm":desc,
            "note":"07's adapters and the new ones differ in GPU/library environment as well as seed"},
         "e0_record":json.load(open(E0_JSON)) if os.path.exists(E0_JSON) else None,
         "env_analysis":ENV,"time":time.strftime("%Y-%m-%d %H:%M:%S")}
    dst=os.path.join(ROOT,"flux_seed_ext_summary.json"); ajson(out,dst,indent=2)
    print(f"[E60] written: {dst}")
if stage("E60"): e60()


## Running it

`C.STAGES` defaults to the full sequence, so **Runtime → Run all** runs E0 → F0 → F1 → F2 → F3 → E60.
Every stage skips completed work. If the session drops, Run all again: the gates re-verify, finished
adapters (`train_meta.json` matching both hashes), generations (per file) and measurements
(per `(tag, gen_idx, K)` row) are skipped. A drop costs at most one adapter's training.

| session | STAGES | wall (A100-80GB reference → G4 estimate) | check |
|---|---|---|---|
| 1 | `("E0","F0")` | ~15 min | four gates PASS; F0 prints 5 finite losses |
| 2 | `("F1",)` | ~4–6 h → ~3–5 h | every ‖lora_B‖ non-zero, comparable to 07's |
| 3 | `("F2",)` | ~6–8 h → ~5–7 h | 3,000 images at 1024, 28 steps |
| 4 | `("F3","E60")` | ~20 min | the Entry 60 reading |

**E0 is not optional.** F1 and F2 refuse to start without a passing `meta/flux_ext_E0.json` for this
exact `CFG_SHA`/`EXT_SHA`.
- *Protocol-hash mismatch:* the notebook names `GRAD_CKPT`/`GEN_BATCH` values that would explain it,
  if any do. Set those in the CFG cell; they are the only fields you may change.
- *Seed-bank check not bit-identical:* expected across GPU generations (A100 → Blackwell) and
  library versions. It passes automatically when the regenerated image correlates ≥ 0.90 with the
  stored one and clearly more than with a different seed. Otherwise, inspect
  `meta/e0_regen_*.png` yourself; set `SEEDBANK_ACCEPT_DRIFT=True` only if the composition is the
  same. Either way the verdict is recorded and goes into the summary JSON. Disclose it.

**E60 will not run early.** It halts unless all twelve adapters × 500 generations × 2 fingerprints are
measured, and unless 07's artefacts still match the E0 snapshot. It first re-derives 07's
three-adapter result (λ_U 0.777 % at n = 3) from 07's own rows. Only then does it compute Entry 60.

**Reading E60.** Quote the construction with every limit: the max-arm λ_U (t₀.₉₉₅,₅ = 4.03) and the
symmetric one-sided 99 % limit answer different questions. Compare the six-adapter FLUX figures only
with SD-3.5 at six adapters per arm (0.249 %, max-arm), never with the k = 12 headline. The old-versus-new
adapter comparison it prints is descriptive only and not part of Entry 60.

**Outputs (all new files):** `adapters/{A,B}_raw_s{3,4,5}_flux/`, `gens/{A,B}_raw_s{3,4,5}_flux/`,
`csv/f3_measure_ext.csv`, `meta/flux_ext_E0.json`, `meta/flux_ext_v1_snapshot.json`,
`meta/e0_regen_*.png`, `flux_seed_ext_summary.json`.
